# 예제 franka_ex07: FR3 Pick & Place

Franka FR3 의 7-DOF 팔과 Franka Hand 그리퍼를 통합해서
가상 물체를 한 위치에서 잡아 다른 위치로 옮기는 시퀀스.
6-DOF 용 `ex07_pick_and_place.py` 를 FR3 워크스페이스에 맞춰 옮겨왔다.

**6-DOF 예제와 다른 점**
- 끝단 링크: `fr3_hand_tcp` (6-DOF 는 `gripper_base`)
- planning group: `fr3_arm`
- 그리퍼: `FollowJointTrajectory` 액션 (6-DOF 는 `GripperCommand`)
- Pick/Place 위치: FR3 reach (~85cm) 기준으로 더 멀리/높게 — Pick `(0.50, 0.00, 0.30)`, Place `(0.00, 0.50, 0.30)`
- 7-DOF redundancy 덕에 같은 pose 도 IK 해가 여러 개 → 더 자연스러운 경로
- `home`(올-제로) 자세가 SRDF 에 없다 → 시작/복귀는 `ready` 사용
- Gazebo Sim 환경 — `use_sim_time=True`

**학습 내용**
- 팔(MoveGroup) + 그리퍼(FollowJointTrajectory) 통합 시퀀스
- IK 시드(현재 조인트)를 활용한 연속 이동 → 과도한 회전 방지
- RViz `MarkerArray` 로 Pick/Place 테이블 시각화
- 단계별 흐름을 노트북 셀로 분리해서 한 셀씩 확인하며 진행

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/pick_place_markers` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다.

Planning Scene Display 도 추가하면 충돌 객체가 같이 보인다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 상수

In [ ]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
# FR3 gripper: fr3_gripper_controller (JointTrajectoryController)
# 액티브 조인트는 fr3_finger_joint1 (prismatic, 0.0=닫힘, 0.04=한쪽 손가락 최대 열림).
# fr3_finger_joint2 는 mimic 이라 컨트롤러에는 안 들어감.
GRIPPER_JOINT     = 'fr3_finger_joint1'
GRIPPER_ACTION    = '/fr3_gripper_controller/follow_joint_trajectory'
GRIPPER_OPEN      = 0.04   # 최대 열림 (m)
GRIPPER_CLOSED    = 0.0    # 닫힘 (m)
MARKER_TOPIC      = '/pick_place_markers'

## 2. ROS 2 초기화 + 노드

In [ ]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from visualization_msgs.msg import MarkerArray, Marker
from std_msgs.msg import ColorRGBA
from control_msgs.action import FollowJointTrajectory
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
from builtin_interfaces.msg import Duration

In [ ]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

In [ ]:
node = Node(
    'franka_ex07_pick_place_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client = ActionClient(node, MoveGroup, 'move_action')

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex07 노트북 노드 생성 완료 ===')
marker_pub = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)

### 2-1. ExecuteTrajectory + Gripper 액션 클라이언트

In [ ]:
execute_client = ActionClient(node, ExecuteTrajectory, 'execute_trajectory')
gripper_client = ActionClient(node, FollowJointTrajectory, GRIPPER_ACTION)

## 3. 서버 / `joint_states` 준비 대기

In [ ]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    if not execute_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('ExecuteTrajectory 액션 서버 연결 실패')
    if not gripper_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('Gripper 액션 서버 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('action server + /joint_states 준비됨')

wait_for_ready()

## 4. SRDF `ready` 자세 읽기

In [ ]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

## 5. Pose / MoveGroup 헬퍼

In [ ]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Point, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

In [ ]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions, MoveItErrorCodes,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only, replan=not plan_only, replan_attempts=3 if not plan_only else 0)
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code_val}')
    return ok

def go_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code_val} (IK 해 없음 가능)')
    return ok

def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

def plan_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                      planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

## 6. ExecuteTrajectory 헬퍼 (계획만 된 궤적 실행용)

`MoveGroup` 액션은 보통 plan+execute 를 한 번에 한다.
Pick & Place 에서는 같은 trajectory 를 RViz 에 미리 시각화한 뒤 실행하고 싶으므로
`PlanningOptions(plan_only=True)` 로 계획만 받고 → 시각화 → `ExecuteTrajectory` 로 실행한다.

In [ ]:
def execute_trajectory(trajectory) -> bool:
    g = ExecuteTrajectory.Goal()
    g.trajectory = trajectory
    sf = execute_client.send_goal_async(g)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return False
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    return rf.result().result.error_code.val == MoveItErrorCodes.SUCCESS

## 7. FK 헬퍼 — 계획된 궤적의 끝단 경로 추출

In [ ]:
from moveit_msgs.srv import GetPositionFK
from moveit_msgs.msg import RobotState

fk_client = node.create_client(GetPositionFK, 'compute_fk')
fk_client.wait_for_service(timeout_sec=10.0)

def trajectory_to_ee_path(trajectory, max_points: int = 60):
    '''RobotTrajectory → fr3_hand_tcp 의 base 기준 좌표 리스트.'''
    jt = trajectory.joint_trajectory
    total = len(jt.points)
    if total == 0:
        return []
    step = max(1, total // max_points)
    indices = list(range(0, total, step))
    if indices[-1] != total - 1:
        indices.append(total - 1)
    pts = []
    for idx in indices:
        req = GetPositionFK.Request()
        req.header.frame_id = REFERENCE_FRAME
        req.fk_link_names = [END_EFFECTOR_LINK]
        rs = RobotState()
        rs.joint_state.name = list(jt.joint_names)
        rs.joint_state.position = list(jt.points[idx].positions)
        req.robot_state = rs
        fut = fk_client.call_async(req)
        rclpy.spin_until_future_complete(node, fut)
        resp = fut.result()
        if resp and resp.error_code.val == MoveItErrorCodes.SUCCESS and resp.pose_stamped:
            p = resp.pose_stamped[0].pose.position
            pts.append((p.x, p.y, p.z))
    return pts

## 8. 그리퍼 헬퍼

In [ ]:
from rclpy.action import ActionClient as _AC
from control_msgs.action import FollowJointTrajectory
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
from builtin_interfaces.msg import Duration

gripper_client = _AC(node, FollowJointTrajectory, GRIPPER_ACTION)
if not gripper_client.wait_for_server(timeout_sec=15.0):
    raise RuntimeError('Gripper(FollowJointTrajectory) 서버 연결 실패')

def move_gripper(position: float, duration_sec: float = 1.0) -> bool:
    '''fr3_finger_joint1 을 position 으로 이동 (0.0=닫힘, 0.04=열림).'''
    pos = float(max(0.0, min(GRIPPER_OPEN, position)))
    g = FollowJointTrajectory.Goal()
    g.trajectory = JointTrajectory()
    g.trajectory.joint_names = [GRIPPER_JOINT]
    pt = JointTrajectoryPoint()
    pt.positions = [pos]
    pt.time_from_start = Duration(sec=int(duration_sec),
                                  nanosec=int((duration_sec - int(duration_sec)) * 1e9))
    g.trajectory.points.append(pt)
    sf = gripper_client.send_goal_async(g)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        node.get_logger().error('gripper goal 거부')
        return False
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    code_val = rf.result().result.error_code
    ok = (code_val == 0)
    node.get_logger().info(f'gripper {pos*1000:.1f}mm 이동 (code={code_val})')
    return ok

def gripper_open():  return move_gripper(GRIPPER_OPEN)
def gripper_close(): return move_gripper(GRIPPER_CLOSED)

## 9. RViz 마커 — Pick/Place 테이블

각 위치를 작은 큐브 + 라벨로 표시. 끝단 경로는 별도 LINE_STRIP 으로 발행.

In [ ]:
COLOR_PICK     = ColorRGBA(r=0.3, g=0.7, b=0.3, a=0.6)   # 초록 (pick)
COLOR_PLACE    = ColorRGBA(r=0.3, g=0.3, b=0.8, a=0.6)   # 파랑 (place)
COLOR_TEXT     = ColorRGBA(r=1.0, g=1.0, b=1.0, a=1.0)
COLOR_EE_PATH  = ColorRGBA(r=1.0, g=0.5, b=0.0, a=0.95)  # 주황 (실행할 경로)

_markers = MarkerArray()

def _publish_all():
    marker_pub.publish(_markers)

def add_table_marker(pos, label: str, idx: int, color: ColorRGBA):
    from geometry_msgs.msg import Point, Vector3
    stamp = node.get_clock().now().to_msg()
    cube = Marker()
    cube.header.frame_id = REFERENCE_FRAME
    cube.header.stamp = stamp
    cube.ns = 'tables'
    cube.id = idx
    cube.type = Marker.CUBE
    cube.action = Marker.ADD
    cube.pose.position = Point(x=pos[0], y=pos[1], z=pos[2] - 0.06)
    cube.pose.orientation.w = 1.0
    cube.scale = Vector3(x=0.12, y=0.12, z=0.10)
    cube.color = color
    text = Marker()
    text.header.frame_id = REFERENCE_FRAME
    text.header.stamp = stamp
    text.ns = 'labels'
    text.id = idx
    text.type = Marker.TEXT_VIEW_FACING
    text.action = Marker.ADD
    text.pose.position = Point(x=pos[0], y=pos[1], z=pos[2] + 0.10)
    text.pose.orientation.w = 1.0
    text.scale.z = 0.05
    text.color = COLOR_TEXT
    text.text = label
    keys = {('tables', idx), ('labels', idx)}
    _markers.markers = [m for m in _markers.markers if (m.ns, m.id) not in keys]
    _markers.markers.extend([cube, text])
    _publish_all()

def publish_ee_path(ee_points, color=COLOR_EE_PATH):
    from geometry_msgs.msg import Point
    if not ee_points:
        return
    stamp = node.get_clock().now().to_msg()
    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = 'ee_path'
    line.id = 0
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.pose.orientation.w = 1.0
    line.scale.x = 0.008
    line.color = color
    line.points = [Point(x=p[0], y=p[1], z=p[2]) for p in ee_points]
    _markers.markers = [m for m in _markers.markers if (m.ns, m.id) != ('ee_path', 0)]
    _markers.markers.append(line)
    _publish_all()

## 10. plan → 시각화 → execute 워크플로

`plan_to_pose_goal` 로 trajectory 만 받고, FK 로 끝단 경로를 추출해 RViz 에 표시한 뒤
`execute_trajectory` 로 실행한다.

In [ ]:
def plan_viz_execute(pose: Pose, vel: float = 0.3, label: str = '') -> bool:
    ok, traj = plan_to_pose_goal(pose, vel=vel, acc=vel, plan_time=10.0)
    if not ok or traj is None:
        node.get_logger().error(f'{label}: 계획 실패')
        return False
    pts = trajectory_to_ee_path(traj)
    if pts:
        publish_ee_path(pts)
    return execute_trajectory(traj)

def plan_viz_execute_joint(joint_values: dict, vel: float = 0.3) -> bool:
    ok, traj = plan_to_joint_goal(joint_values, vel=vel, acc=vel)
    if not ok or traj is None:
        return False
    pts = trajectory_to_ee_path(traj)
    if pts:
        publish_ee_path(pts)
    return execute_trajectory(traj)

## 11. Pick / Place 위치 정의 + RViz 표시

FR3 워크스페이스에 맞게 base 로부터 ~50cm 전방 / 좌측 ~50cm 에 둔다.
TCP 는 아래를 향하도록 `roll = π`.

In [ ]:
pick_pos  = (0.50, 0.00, 0.30)
place_pos = (0.00, 0.50, 0.30)
APPROACH_HEIGHT = 0.45   # 들어올릴 때 / 운반 중 z

add_table_marker(pick_pos,  'Pick',  0, COLOR_PICK)
add_table_marker(place_pos, 'Place', 1, COLOR_PLACE)
node.get_logger().info('Pick / Place 마커 발행')

## 12. 1단계 — `ready` 자세 + 그리퍼 열기

In [ ]:
node.get_logger().info('--- 1단계: ready 자세 + 그리퍼 열기 ---')
plan_viz_execute_joint(ready_target, vel=0.3)
gripper_open()
time.sleep(1.0)

## 13. 2단계 — Pick 위치 위로 접근 (`approach`)

바로 Pick 위치로 가지 말고 같은 X-Y 위에서 약 15cm 위로 먼저 간다.
이러면 마지막 하강을 직선에 가깝게 만들어 충돌 위험을 줄일 수 있다.

In [ ]:
approach_pick = make_pose(pick_pos[0], pick_pos[1], APPROACH_HEIGHT,
                          math.pi, 0.0, 0.0)
node.get_logger().info('--- 2단계: Pick 접근 자세 ---')
plan_viz_execute(approach_pick, label='ApproachPick')
time.sleep(0.5)

## 14. 3단계 — Pick 위치로 하강

In [ ]:
pick_pose = make_pose(*pick_pos, math.pi, 0.0, 0.0)
node.get_logger().info('--- 3단계: Pick 위치 하강 ---')
plan_viz_execute(pick_pose, vel=0.2, label='Pick')
time.sleep(0.5)

## 15. 4단계 — 그리퍼 닫기 (가상 물체 잡기)

In [ ]:
node.get_logger().info('--- 4단계: 그리퍼 닫기 ---')
gripper_close()
time.sleep(1.0)

## 16. 5단계 — 들어올리기

In [ ]:
lift = make_pose(pick_pos[0], pick_pos[1], APPROACH_HEIGHT,
                 math.pi, 0.0, 0.0)
node.get_logger().info('--- 5단계: 물체 들어올리기 ---')
plan_viz_execute(lift, vel=0.2, label='Lift')
time.sleep(0.5)

## 17. 6단계 — Place 위치 위로 운반

운반 중에는 yaw 를 90° 돌려 손목 방향이 자연스럽게 따라가도록 한다.

In [ ]:
approach_place = make_pose(place_pos[0], place_pos[1], APPROACH_HEIGHT,
                           math.pi, 0.0, math.pi / 2)
node.get_logger().info('--- 6단계: Place 위치 위로 운반 ---')
plan_viz_execute(approach_place, label='Transport')
time.sleep(0.5)

## 18. 7단계 — Place 위치로 하강

In [ ]:
place_pose = make_pose(*place_pos, math.pi, 0.0, math.pi / 2)
node.get_logger().info('--- 7단계: Place 하강 ---')
plan_viz_execute(place_pose, vel=0.2, label='Place')
time.sleep(0.5)

## 19. 8단계 — 그리퍼 열기 (놓기) + 후퇴

In [ ]:
node.get_logger().info('--- 8단계: 그리퍼 열기 (놓기) ---')
gripper_open()
time.sleep(1.0)

retreat = make_pose(place_pos[0], place_pos[1], APPROACH_HEIGHT,
                    math.pi, 0.0, math.pi / 2)
node.get_logger().info('--- 후퇴 ---')
plan_viz_execute(retreat, label='Retreat')
time.sleep(0.5)

## 20. 9단계 — `ready` 복귀

In [ ]:
node.get_logger().info('--- 9단계: ready 복귀 ---')
plan_viz_execute_joint(ready_target, vel=0.3)
node.get_logger().info('=== franka_ex07 완료! ===')

## 21. 정리

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass